# 端到端示例 1：Reaction-to-Assay

目标：从任务说明开始，规划一条合法反应实验，执行到 final assay，读取谱图和指标，并写出下一轮实验判断。

本 notebook 是完整流程模板，不是最高分策略。重点是可验证、可复现、可解释。

## 1. 任务规划

使用 `task_prompt()`、`available_actions()` 和 `action_schema()` 先读 public contract。不要直接猜 payload 字段名。

In [ ]:
from __future__ import annotations

from pathlib import Path

import gymnasium as gym
import pandas as pd
from IPython.display import display

import chemworld  # registers ChemWorld

OUTPUT_DIR = Path("runs/end_to_end")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 4)

env = gym.make("ChemWorld", task_id="reaction-to-assay", seed=0)
obs, info = env.reset(seed=0)
display(env.unwrapped.task_prompt())
display(pd.DataFrame(env.unwrapped.available_actions()).head(12))
display(env.unwrapped.action_schema("heat"))
env.close()


## 2. 设计并验证 recipe

这条 recipe 先做中间 HPLC，再淬灭、终止并执行 final assay。

In [ ]:
recipe = [
    {"operation": "add_solvent", "volume_L": 0.030, "solvent": 1},
    {"operation": "add_reagent", "amount_mol": 0.012},
    {"operation": "add_catalyst", "catalyst": 2, "catalyst_amount_mol": 0.0004},
    {"operation": "heat", "target_temperature_K": 350.0, "duration_s": 1200.0, "stirring_speed_rpm": 800.0},
    {"operation": "sample", "sample_volume_L": 0.0005},
    {"operation": "measure", "instrument": "hplc"},
    {"operation": "quench"},
    {"operation": "terminate"},
    {"operation": "measure", "instrument": "final_assay"},
]

validation_rows = []
env = gym.make("ChemWorld", task_id="reaction-to-assay", seed=0)
env.reset(seed=0)
for action in recipe:
    validation = env.unwrapped.validate_action(action)
    validation_rows.append({"operation": action["operation"], "valid": validation["valid"], "reasons": validation.get("invalid_reasons", [])})
    if validation["valid"]:
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            break
env.close()
display(pd.DataFrame(validation_rows))


## 3. 执行实验并保存轨迹表

`reward` 是在线观测分数；正式榜单看 final assay 的 `leaderboard_score`。

In [ ]:
def run_recipe(task_id: str, recipe: list[dict], *, seed: int = 0) -> tuple[pd.DataFrame, dict]:
    env = gym.make("ChemWorld", task_id=task_id, seed=seed)
    obs, info = env.reset(seed=seed)
    rows = []
    final_info = info
    try:
        for step, action in enumerate(recipe, start=1):
            validation = env.unwrapped.validate_action(action)
            obs, reward, terminated, truncated, info = env.step(action)
            final_info = info
            rows.append(
                {
                    "step": step,
                    "operation": action["operation"],
                    "valid_before_step": validation["valid"],
                    "invalid_reasons": "; ".join(validation.get("invalid_reasons", [])),
                    "reward": float(reward),
                    "leaderboard_score": info.get("leaderboard_score"),
                    "precondition_failed": info.get("constraint_flags", {}).get("precondition_failed"),
                    "unsafe": info.get("constraint_flags", {}).get("unsafe"),
                    "cost": info.get("cost"),
                    "observed_keys": ", ".join(info.get("observed_keys", [])),
                    "has_raw_signal": bool(info.get("raw_signal")),
                    "terminated": terminated,
                    "truncated": truncated,
                }
            )
            if terminated or truncated:
                break
    finally:
        env.close()
    return pd.DataFrame(rows), final_info


def spectra_frame(final_info: dict, channel: str = "hplc") -> pd.DataFrame:
    packet = final_info.get("raw_signal", {})
    spectra = packet.get("spectra", {}) if isinstance(packet, dict) else {}
    signal = spectra.get(channel, {})
    if channel in {"hplc", "gc"}:
        return pd.DataFrame({"x": signal.get("time_min", []), "signal": signal.get("intensity", [])})
    if channel == "uvvis":
        return pd.DataFrame({"x": signal.get("wavelength_nm", []), "signal": signal.get("absorbance", [])})
    return pd.DataFrame()

rows, final_info = run_recipe("reaction-to-assay", recipe, seed=0)
display(rows)
rows.to_csv(OUTPUT_DIR / "reaction_to_assay_trace.csv", index=False)


## 4. 谱图与指标

final assay 返回多通道 packet。这里读取 HPLC 和 UV-vis，用于确认 processed metrics 是否有观测依据。

In [ ]:
display(final_info.get("processed_estimate", {}))

hplc = spectra_frame(final_info, "hplc")
uvvis = spectra_frame(final_info, "uvvis")
if not hplc.empty:
    display(hplc.head())
    hplc.plot(x="x", y="signal", title="Final assay HPLC signal", xlabel="time / min")
if not uvvis.empty:
    uvvis.plot(x="x", y="signal", title="Final assay UV-vis signal", xlabel="wavelength / nm")


## 5. 反思记录

请把下面字段填完整，再把 CSV、图和结论一起作为本次实验 artifact。

In [ ]:
reflection = {
    "best_observed_step": rows.sort_values("reward", ascending=False).iloc[0].to_dict(),
    "chemical_hypothesis": "示例：当前温度产生了可测产物，但选择性仍有限，下一轮应比较更短反应时间。",
    "next_experiment": {"target_temperature_K": 340.0, "duration_s": 900.0, "reason": "降低副反应和风险"},
    "reproducibility_evidence": str(OUTPUT_DIR / "reaction_to_assay_trace.csv"),
}
reflection
